In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")


In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

You have leftover chicken and rice! Here are a few ideas for what you can make:

1.  **One-Pot Chicken and Rice:** This is a comforting and simple dish where chicken and rice cook together in a single pot. Some recipes include vegetables like broccoli and can be flavored with herbs, lemon, or even cream of chicken soup for extra richness.

2.  **Easy Chicken and Rice Skillet:** Similar to the one-pot method, this dish often features tender chicken thighs and fluffy rice simmered in broth, topped with a fresh gremolata (a mix of lemon zest, parsley, and garlic).

3.  **15-Minute Chicken & Rice Dinner:** If you're short on time, this recipe combines chicken, rice, and veggies in one skillet and can be ready in just 15 minutes. It often uses condensed cream of chicken soup and instant rice.

4.  **Old Fashioned Chicken and Rice:** A hearty Southern comfort food classic, this recipe typically involves chicken, rice, celery, and onions steamed together in one pot.

5.  **Marry Me Chicken an

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='b86d05f5-ebc5-42cd-a778-2cf6fa974838'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "recipes with chicken and rice"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019da6ab-2bb7-78e1-8d0a-7ab0938a2845-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes with chicken and rice'}, 'id': '73192dec-46d1-4a78-b8d3-4daa8fff8231', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 19, 'total_tokens': 133, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='{"query": "recipes with chicken and rice", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://i